# Patent Disruption — Granted vs Extended (granted + application) network

Compares `patent_disruption.parquet` (**granted** citation network) against
`patent_disruption_app.parquet` (**extended** = granted + application/pre-grant citations, cited pgpub mapped
to patent via `pg_granted_pgpubs_crosswalk`). Both share the schema `patent_id` + `{CD,F,E,G,ni,nj,nk}` x
window `{_3,_5,_10,_all}`. Main window = **5**. Adding application citations densifies the network (more
citers of a focal patent and of its references), so `ni/nj/nk` should rise; the sign/size of the CD shift
is the question.

In [ ]:
import os, sys, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PatentView')
import pv_common as pv
ROOT, D, OUT = pv.BASE, pv.GRANTED, pv.OUT
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
POUT = pv.OUT
WSFX = ['_3', '_5', '_10', '_all']
MET = ['CD', 'F', 'E', 'G', 'ni', 'nj', 'nk']

## 1. Summary table — means by window (base vs extended)

In [ ]:
rows = []
for s in WSFX:
    for m in MET:
        rows.append({'window': s, 'metric': m,
                     'base_mean': base[f'{m}{s}'].mean(), 'app_mean': app[f'{m}{s}'].mean(),
                     'base_defined%': 100*base[f'{m}{s}'].notna().mean(),
                     'app_defined%': 100*app[f'{m}{s}'].notna().mean()})
tab = pd.DataFrame(rows)
tab['delta'] = tab['app_mean'] - tab['base_mean']
pd.set_option('display.float_format', lambda v: f'{v:.4f}')
display(tab[tab['metric'] == 'CD'])
display(tab[tab['metric'].isin(['ni', 'nj', 'nk'])])
display(tab[tab['metric'].isin(['F', 'E', 'G'])])

## 2. Window 5 — CD shift (extended − granted)

In [ ]:
j = base[['patent_id', 'CD_5', 'ni_5', 'nj_5', 'nk_5']].merge(
    app[['patent_id', 'CD_5', 'ni_5', 'nj_5', 'nk_5']], on='patent_id', suffixes=('_b', '_a'))
d = j.dropna(subset=['CD_5_b', 'CD_5_a'])
dd = d['CD_5_a'] - d['CD_5_b']
print(f'patents with CD_5 defined in both: {len(d):,}')
print(f'mean CD_5  base {d["CD_5_b"].mean():.4f} -> app {d["CD_5_a"].mean():.4f}  (delta {dd.mean():+.4f})')
print(f'more consolidating (CD down): {100*(dd<0).mean():.1f}%   more disruptive (CD up): {100*(dd>0).mean():.1f}%   '
      f'unchanged: {100*(dd==0).mean():.1f}%')
print('Pearson  CD_5 base vs app:', round(d[['CD_5_b', 'CD_5_a']].corr().iloc[0, 1], 4))
print('Spearman CD_5 base vs app:', round(d[['CD_5_b', 'CD_5_a']].corr(method='spearman').iloc[0, 1], 4))
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].hist(dd.values, bins=120, color='#8172b3'); ax[0].axvline(0, color='k', lw=0.8)
ax[0].set_title('Delta CD_5 = extended - granted'); ax[0].set_xlabel('delta CD_5'); ax[0].set_yscale('log')
smp = d.sample(min(200000, len(d)), random_state=0)
ax[1].hexbin(smp['CD_5_b'], smp['CD_5_a'], gridsize=60, bins='log', cmap='viridis')
ax[1].plot([-1, 1], [-1, 1], 'r--', lw=1); ax[1].set_xlabel('CD_5 granted'); ax[1].set_ylabel('CD_5 extended')
ax[1].set_title('CD_5 base vs extended (hexbin, log)')
means = pd.DataFrame({'granted': [j['ni_5_b'].mean(), j['nj_5_b'].mean(), j['nk_5_b'].mean()],
                      'extended': [j['ni_5_a'].mean(), j['nj_5_a'].mean(), j['nk_5_a'].mean()]}, index=['ni', 'nj', 'nk'])
means.plot.bar(ax=ax[2], color=['#333333', '#e377c2']); ax[2].set_title('Mean ni/nj/nk (window 5)')
ax[2].set_ylabel('mean count'); ax[2].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 3. Window 5 — CD by CPC section (base vs extended)

In [ ]:
m5 = base[['patent_id', 'CD_5']].rename(columns={'CD_5': 'CD_b'}).merge(
     app[['patent_id', 'CD_5']].rename(columns={'CD_5': 'CD_a'}), on='patent_id').merge(
     meta[['patent_id', 'cpc_section']], on='patent_id')
m5 = m5[m5['cpc_section'].isin(list('ABCDEFGHY'))]
gs = m5.groupby('cpc_section').agg(CD_b=('CD_b', 'mean'), CD_a=('CD_a', 'mean'), n=('CD_b', 'size'))
gs['delta'] = gs['CD_a'] - gs['CD_b']
display(gs.round(4))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
gs[['CD_b', 'CD_a']].plot.bar(ax=ax[0], color=['#333333', '#e377c2'])
ax[0].set_title('Mean CD_5 by CPC section'); ax[0].set_ylabel('mean CD_5'); ax[0].axhline(0, color='k', lw=0.6)
ax[0].legend(['granted', 'extended']); ax[0].tick_params(axis='x', rotation=0)
gs['delta'].plot.bar(ax=ax[1], color='#8172b3'); ax[1].axhline(0, color='k', lw=0.6)
ax[1].set_title('Delta CD_5 (extended - granted) by CPC section'); ax[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 4. CD trend by grant year (base vs extended, window 5)

In [ ]:
tr = base[['patent_id', 'CD_5']].rename(columns={'CD_5': 'CD_b'}).merge(
     app[['patent_id', 'CD_5']].rename(columns={'CD_5': 'CD_a'}), on='patent_id').merge(
     meta[['patent_id', 'grant_year']], on='patent_id')
tr = tr[tr['grant_year'].between(1990, 2020)]
g = tr.groupby('grant_year')[['CD_b', 'CD_a']].mean()
fig, ax = plt.subplots(figsize=(9, 4.5))
g['CD_b'].plot(ax=ax, color='#333333', label='granted'); g['CD_a'].plot(ax=ax, color='#e377c2', label='extended')
ax.axhline(0, color='k', lw=0.6); ax.set_title('Mean CD_5 by grant year: granted vs extended network')
ax.set_ylabel('mean CD_5'); ax.legend(); plt.tight_layout(); plt.show()
print('Application citations exist mainly for grant years >= ~2002 (pre-grant publication era),')
print('so the base/extended gap should widen from the early 2000s onward.')